<a href="https://colab.research.google.com/github/Rahma-Shaban/Intelligent-Educational-Assessment-System/blob/main/notebooks/04_student_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv("exam_results_rows.csv")

In [3]:
df.head()

,id,student_name,student_phone,total_score,correct_answers_count,total_questions,percentage,answers,created_at,exam_title,allow_retake
0,0051a17a-3885-4f04-89c4-30a435a3f65c,احمد بلال,01102189968,532,10,10,100,"{""0"":""الحواسيب الكبيرة ← الحواسيب الشخصية ← ال...",2026-09-06 09:38:55.998528+00,اختبار تقييم المستوى,True
1,0193562c-d67f-4dd0-bc22-fa3b9cb0c35f,زياد احمد محمد احمد حسن الجاويش,01515816595,702,14,15,93,"{""0"":""التعلم من البيانات لاكتشاف الأنماط"",""1"":...",2026-09-06 21:07:09.801478+00,اختبار الدرس الثاني - الذكاء الأصطناعي,True
2,054fc5c9-5385-4d47-abd2-61e7a117f45a,علياء محمد كمال,01019896490,671,12,15,80,"{""0"":""تحليل الأنماط المتكررة في بيانات السلوك ...",2026-09-13 21:06:09.579523+00,اختبار الدرس الثالث - تطبيقات الذكاء الأصطناعي,True
3,05656ad6-d61e-43fe-8a78-54763144524b,مريم ابراهيم يوسف,01202968045,525,10,15,67,"{""0"":""التعلم من البيانات لاكتشاف الأنماط"",""1"":...",2026-09-06 21:06:49.510455+00,اختبار الدرس الثاني - الذكاء الأصطناعي,True
4,05c37c1a-c360-48c9-91e8-211aaa76fd06,محمد محمود محمد,01104619589,677,12,15,80,"{""0"":""التعلم من البيانات لاكتشاف الأنماط"",""1"":...",2026-09-07 20:55:12.821267+00,اختبار الدرس الثاني - الذكاء الأصطناعي,True


In [4]:
import numpy as np
import pandas as pd

df = pd.read_csv("exam_results_rows.csv")

df["student_phone"] = (
    df["student_phone"].astype(str).str.replace(r"\s+", "", regex=True)
)

student_features = (
    df.groupby("student_phone")
    .agg(
        student_name=(
            "student_name",
            lambda x: x.mode()[0] if not x.empty else "",
        ),
        exams_taken=("exam_title", "nunique"),
        avg_percentage=("percentage", "mean"),
        max_percentage=("percentage", "max"),
        min_percentage=("percentage", "min"),
        std_percentage=("percentage", "std"),
    )
    .reset_index()
)

student_features["std_percentage"] = student_features["std_percentage"].fillna(
    0
)

filtered_students = student_features[
    student_features["exams_taken"] > 1
].copy()

filtered_students["avg_percentage"] = filtered_students[
    "avg_percentage"
].round(1)
filtered_students["std_percentage"] = filtered_students[
    "std_percentage"
].round(2)

filtered_students.to_csv(
    "students_for_clustering.csv", index=False, encoding="utf-8-sig"
)
print(f"done")

done


In [5]:
import numpy as np
import pandas as pd

df = pd.read_csv("/content/students_for_clustering.csv")

In [6]:
df.head()

,student_phone,student_name,exams_taken,avg_percentage,max_percentage,min_percentage,std_percentage
0,1007729781,روان محمود حسن محمد يوسف,6,84.5,100,73,9.22
1,1008836765,محمد علي,4,55.8,73,33,18.03
2,1011755294,شروق حسن السيد,5,82.0,100,47,20.84
3,1015804169,عبدالله صلاح احمد,5,84.6,93,67,11.19
4,1019896490,علياء محمد كمال,3,89.0,100,80,10.15


In [7]:
from sklearn.preprocessing import StandardScaler

features = [
    'avg_percentage',
    'max_percentage',
    'min_percentage',
    'exams_taken',
    'std_percentage',
]
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Scaling done')

Scaling done


In [8]:
import joblib
from sklearn.cluster import KMeans
X_scaled[:, 0] = X_scaled[:, 0] * 1.5
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

In [9]:
cluster_order = (
    df.groupby('cluster')['avg_percentage']
    .mean()
    .sort_values(ascending=True)
    .index
)

cluster_numeric_mapping = {
    cluster_order[0]: 0,  # needs support
    cluster_order[1]: 1,  # intermediate
    cluster_order[2]: 2,  # Advanced
}


df['level_code'] = df['cluster'].map(cluster_numeric_mapping)


level_labels = {
    0: 'Needs Support',
    1: 'Intermediate',
    2: 'Advanced',
}
df['student_level'] = df['level_code'].map(level_labels)

display(
    df[
        [
            'student_name',
            'avg_percentage',
            'exams_taken',
            'level_code',
            'student_level',
        ]
    ]
)

,student_name,avg_percentage,exams_taken,level_code,student_level
0,روان محمود حسن محمد يوسف,84.5,6,2,Advanced
1,محمد علي,55.8,4,0,Needs Support
2,شروق حسن السيد,82.0,5,1,Intermediate
3,عبدالله صلاح احمد,84.6,5,2,Advanced
4,علياء محمد كمال,89.0,3,2,Advanced
5,ايمان محمد محمد علي,90.0,2,2,Advanced
6,شيماء محمود حسن,50.0,3,0,Needs Support
7,ياسمين محمد حسن,87.8,6,2,Advanced
8,حبيبه محمد حسن,77.4,5,1,Intermediate
9,سما عبدالفتاح علي,81.5,4,2,Advanced


In [10]:
import joblib

# حفظ كل ملحقات النموذج في ملف بايبلاين واحد
pipeline_data = {
    'scaler': scaler,
    'model': kmeans,
    'features': features,
    'avg_weight': 1.5,
    'numeric_mapping': cluster_numeric_mapping,
    'level_labels': level_labels,
}

joblib.dump(pipeline_data, 'student_cluster_pipeline.pkl')
print('تم حفظ student_cluster_pipeline.pkl بنجاح!')

تم حفظ student_cluster_pipeline.pkl بنجاح!
